# 01 区间分割法与二分法

本节从最稳健的标量求根方法开始。若函数 $f$ 在区间 $[a,b]$ 上连续，且

$$
f(a)f(b)<0,
$$

则介值定理保证区间内至少存在一个根。二分法每次把区间一分为二，并保留仍然变号的半区间，因此始终保持“根在括区间内”这个不变量。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import bisection_method, find_sign_change_brackets


## 根的隔离和变号区间扫描

实际问题中常常不知道根在哪里。一个保守做法是先在大区间上取网格，寻找函数值变号的小区间。这个方法能找到有变号的根，但可能漏掉偶重根，也可能在网格过粗时漏掉靠得很近的多个根。


In [ ]:
def teaching_find_brackets(f, a, b, subintervals):
    x = np.linspace(a, b, subintervals + 1)
    y = np.array([f(float(xi)) for xi in x])
    brackets = []
    for i in range(subintervals):
        if y[i] == 0:
            brackets.append((x[i], x[i]))
        elif y[i] * y[i + 1] < 0:
            brackets.append((x[i], x[i + 1]))
    if y[-1] == 0:
        brackets.append((x[-1], x[-1]))
    return brackets

f = lambda x: (x - 1.0) * (x + 0.5) * (x - 2.0)
brackets = teaching_find_brackets(f, -1.0, 2.5, 35)
print(brackets)
print(find_sign_change_brackets(f, -1.0, 2.5, 35))


## 二分法算法

二分法的步骤是：

1. 检查 $f(a)f(b)<0$；
2. 取中点 $m=(a+b)/2$；
3. 若 $f(a)f(m)<0$，令新区间为 $[a,m]$；否则令新区间为 $[m,b]$；
4. 重复直到区间长度或残差满足停止准则。

第 $k$ 次迭代后，区间长度不超过

$$
\frac{b-a}{2^k},
$$

中点误差不超过区间长度的一半。


In [ ]:
def teaching_bisection(f, a, b, tolerance=1e-10, max_iterations=100):
    fa = f(a)
    fb = f(b)
    if fa * fb > 0:
        raise ValueError("二分法需要变号区间")
    history = []
    left, right = a, b
    f_left = fa
    for k in range(1, max_iterations + 1):
        mid = 0.5 * (left + right)
        f_mid = f(mid)
        history.append(mid)
        if abs(f_mid) <= tolerance or 0.5 * (right - left) <= tolerance:
            return mid, np.array(history)
        if f_left * f_mid < 0:
            right = mid
        else:
            left = mid
            f_left = f_mid
    return mid, np.array(history)

root, history = teaching_bisection(lambda x: math.cos(x) - x, 0.0, 1.0, tolerance=1e-12)
print(root, len(history), math.cos(root) - root)


## 与正式实现对照

正式实现返回统一结果对象，包括根、迭代次数、收敛标记、残差和历史中点。教学实现更短，便于看清区间更新逻辑。


In [ ]:
result = bisection_method(lambda x: math.cos(x) - x, 0.0, 1.0, tolerance=1e-12)
print(result)
print("教学/正式根差异：", abs(root - result.root))


## 误差上界实验

二分法的误差上界只依赖初始区间长度，不依赖导数。下面比较实际误差和理论上界。


In [ ]:
exact = 0.7390851332151607
errors = np.abs(history - exact)
bounds = 1.0 / (2.0 ** np.arange(1, history.size + 1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(errors, "o-", label="实际误差")
ax.semilogy(bounds, "--", label="区间上界")
ax.set_xlabel("迭代次数")
ax.set_ylabel("误差")
ax.set_title("二分法误差上界")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 偶重根的局限

若根是偶重根，例如

$$
f(x)=(x-1)^2,
$$

函数在根两侧不变号。单纯依赖变号扫描会漏掉这种根。实际处理中可以结合网格最小值、导数信息、函数图像或其它开方法。


In [ ]:
even = lambda x: (x - 1.0) ** 2
x = np.linspace(0.0, 2.0, 101)
y = even(x)
print("变号区间：", find_sign_change_brackets(even, 0.0, 2.0, 50))
print("网格最小值位置：", x[np.argmin(y)])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("偶重根没有变号")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.grid(True, alpha=0.3)
plt.show()


## 本节小结

区间扫描和二分法的优点是可靠、简单、可给出误差上界；缺点是收敛慢，并且依赖连续性和变号条件。二分法适合在复杂方法之前提供安全初值，也适合作为 Newton 或弦截法失败后的保底策略。后续开方法会提高速度，但需要更细致的失败保护。
